# Step 3 — Production Load

This notebook:

1. Connects to the database
2. Creates the production table with constraints
3. Loads transformed data from Step 2
4. Runs validation checks

In [7]:
from sqlalchemy import create_engine, text
import os

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "layereddb")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")

print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)
print("PASS SET:", bool(DB_PASS))

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("✅ SELECT 1 =", conn.execute(text("SELECT 1")).scalar())

HOST: localhost
PORT: 5433
DB: layereddb
USER: khizra_tariq
PASS SET: True
✅ SELECT 1 = 1


In [10]:
CREATE_SQL = """
CREATE SCHEMA IF NOT EXISTS berlin_source_data;

CREATE TABLE IF NOT EXISTS berlin_source_data.gas_stations (
    id VARCHAR(20) PRIMARY KEY,
    district_id VARCHAR(20) NOT NULL,
    name VARCHAR(200) NOT NULL DEFAULT 'Unknown',
    brand VARCHAR(100),
    fuel_types TEXT,
    is_24h BOOLEAN DEFAULT FALSE,
    address VARCHAR(200),
    postal_code VARCHAR(10),
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    geometry VARCHAR,
    neighborhood VARCHAR(100),
    district VARCHAR(100),
    neighborhood_id VARCHAR(20),
    amenities VARCHAR(255),

    CONSTRAINT coord_check CHECK (
        latitude BETWEEN 52.3 AND 52.7
        AND longitude BETWEEN 13.0 AND 13.8
    )
);
"""

with engine.connect() as conn:
    for stmt in CREATE_SQL.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
    conn.commit()

print("✅ Table berlin_source_data.gas_stations created (no FK yet)")

✅ Table berlin_source_data.gas_stations created (no FK yet)


In [11]:
sql = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
"""
with engine.connect() as conn:
    rows = conn.execute(text(sql)).fetchall()

len(rows), rows[:50]

(2, [('berlin_source_data', 'gas_stations'), ('gas_stations', 'stations')])

In [14]:
import pandas as pd
from pathlib import Path

csv_path = Path("..") / "sources" / "gas_stations_transformed.csv"
df = pd.read_csv(csv_path)

# Reviewer: remove ":" from columns
df.columns = [c.replace(":", "_") for c in df.columns]

print("rows:", len(df))
print("cols:", df.columns.tolist())
df.head(3)

rows: 1844
cols: ['id', 'name', 'brand', 'operator', 'address', 'latitude', 'longitude', 'district', 'district_id', 'neighborhood', 'neighborhood_id', 'opening_hours', 'car_wash', 'compressed_air', 'wheelchair', 'fuel_diesel', 'fuel_octane_95', 'geometry_wkt']


,id,name,brand,operator,address,latitude,longitude,district,district_id,neighborhood,neighborhood_id,opening_hours,car_wash,compressed_air,wheelchair,fuel_diesel,fuel_octane_95,geometry_wkt
0,16541597,Aral,Aral,NaN,"Aral, Seestraße, Brüsseler Kiez, Wedding, Mitt...",52.546315,13.345599,Mitte,11001001,Wedding,105,24/7,NaN,yes,no,yes,retail,POINT (13.345599 52.546315)
1,26756830,Aral,Aral,Mike Seitz,"Aral, 55, Beusselstraße, Beusselkiez, Moabit, ...",52.530694,13.328225,Mitte,11001001,Moabit,102,24/7,NaN,yes,yes,yes,yes,POINT (13.328225 52.530694)
2,26867411,Bavaria petrol,NaN,Bavaria Petrol,"Bavaria petrol, 14, Heilbronner Straße, Witzle...",52.501974,13.294496,Charlottenburg-Wilmersdorf,11004004,Halensee,407,Mo-Fr 07:00-20:00; Sa 07:00-18:00,NaN,yes,no,yes,yes,POINT (13.294496 52.501974)


In [15]:
required = [
    "id", "district_id", "name",
    "latitude", "longitude",
    "district", "neighborhood", "neighborhood_id"
]

missing = [c for c in required if c not in df.columns]
print("Missing required columns:", missing)

print("Has geometry column?", "geometry" in df.columns)

Missing required columns: []
Has geometry column? False


In [18]:
# Create df_prod from df (Step 2 output)

df2 = df.copy()

# Ensure correct types
df2["id"] = df2["id"].astype(str)
df2["name"] = df2["name"].fillna("Unknown").replace("", "Unknown")

# Add missing production columns if they don't exist
for col in ["fuel_types", "is_24h", "postal_code", "amenities", "geometry"]:
    if col not in df2.columns:
        df2[col] = None

# Ensure these exist
for col in ["brand", "address", "district", "district_id", "neighborhood", "neighborhood_id"]:
    if col not in df2.columns:
        df2[col] = None

# Final production order (must match CREATE TABLE)
df_prod = df2[
    [
        "id",
        "district_id",
        "name",
        "brand",
        "fuel_types",
        "is_24h",
        "address",
        "postal_code",
        "latitude",
        "longitude",
        "geometry",
        "neighborhood",
        "district",
        "neighborhood_id",
        "amenities",
    ]
].copy()

print("✅ df_prod created:", df_prod.shape)

✅ df_prod created: (1844, 15)


In [19]:
df_prod.head(1)

,id,district_id,name,brand,fuel_types,is_24h,address,postal_code,latitude,longitude,geometry,neighborhood,district,neighborhood_id,amenities
0,16541597,11001001,Aral,Aral,None,None,"Aral, Seestraße, Brüsseler Kiez, Wedding, Mitt...",None,52.546315,13.345599,None,Wedding,Mitte,105,None


In [21]:
import pandas as pd

def to_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "yes", "y", "24/7", "24h"}

df_prod["is_24h"] = df_prod["is_24h"].apply(to_bool).astype(bool)

print(df_prod["is_24h"].value_counts(dropna=False).head())
print("dtype:", df_prod["is_24h"].dtype)

is_24h
False    1844
Name: count, dtype: int64
dtype: bool


In [22]:
from sqlalchemy import text

target_schema = "berlin_source_data"
target_table = "gas_stations"
staging_table = "gas_stations_staging"

# 1) Load into staging (OK to replace staging)
df_prod.to_sql(
    staging_table,
    engine,
    schema=target_schema,
    if_exists="replace",
    index=False,
    method="multi"
)
print("✅ Loaded into staging")

# 2) Upsert into production table (keeps constraints)
cols = df_prod.columns.tolist()
col_list = ", ".join(cols)

update_cols = [c for c in cols if c != "id"]
set_clause = ", ".join([f"{c}=EXCLUDED.{c}" for c in update_cols])

UPSERT_SQL = f"""
INSERT INTO {target_schema}.{target_table} ({col_list})
SELECT {col_list}
FROM {target_schema}.{staging_table}
ON CONFLICT (id) DO UPDATE SET
{set_clause};

DROP TABLE IF EXISTS {target_schema}.{staging_table};
"""

with engine.connect() as conn:
    conn.execute(text(UPSERT_SQL))
    conn.commit()

print(f"🚀 Upsert complete. Rows processed: {len(df_prod)}")

✅ Loaded into staging
🚀 Upsert complete. Rows processed: 1844


In [23]:
from sqlalchemy import text

VALIDATION_SQL = """
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT id) AS distinct_ids
FROM berlin_source_data.gas_stations;
"""

with engine.connect() as conn:
    result = conn.execute(text(VALIDATION_SQL)).fetchone()

print("Total rows:", result[0])
print("Distinct IDs:", result[1])
print("Duplicates:", result[0] - result[1])

Total rows: 1844
Distinct IDs: 1844
Duplicates: 0
